### 1. Reading libarires and getting the params (Catalog,schema, and API key)

In [0]:
import requests
from datetime import datetime, timezone
from urllib.parse import urlencode
import time
import json
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)


catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
api_key = dbutils.secrets.get(
    catalog = catalog,
    schema= schema,
    key="owm_api_token"
)

assert api_key, "Secret is empty"
print("Secret read successfully")


## 2. Fetch Weather Forecast Data

This step collects weather forecasts for London, New York, Tokyo, Sydney, and Paris from the OpenWeather API.

### Request and Retry Logic

- Call the forecast endpoint for each city using the API key retrieved securely.
- Apply a 30-second timeout to each request.
- Retry HTTP **429** and **5xx** responses up to three times, waiting **1, 2, and 4 seconds** between attempts.
- Store the final response after the retries finish.
- Since no `units` parameter is supplied, temperatures use the default **Kelvin** unit.

### Prepare Bronze Records

Capture the following fields for each city:

| Column | Description |
|---|---|
| `city` | Requested city and country code, including the state code where supplied. |
| `request_url` | Request URL excluding the API key. |
| `http_status` | HTTP status code from the final response. |
| `raw_payload` | Response body serialized as a JSON string, or preserved as text if it is not valid JSON. |
| `ingestion_timestamp` | UTC timestamp recorded when the response is collected. |

Finally, use `zip()` to combine the collected values into `bronze_record`, a list of dictionaries containing one record per city, ready to write to the Bronze table.

**Note:** This code retries HTTP responses only. Network errors and request timeouts are not retried by this loop.

In [0]:
url = "https://api.openweathermap.org/data/2.5/forecast"
city , request_url, http_status, raw_payload,ingestion_timestamp = [], [], [], [],[]
cities = [
    "London,uk",
    "New York,NY,us",
    "Tokyo,jp",
    "Sydney,au",
    "Paris,fr"
]

# loop through cities 
for location in cities:
    # defining the parameters to pass to API
    params = {
        "q": location,
        "appid": api_key,
    }

    safe_url = url + "?" + urlencode({
        "q": location    })

    response = requests.get(url, params=params, timeout=30)

    retry_count = 0
    max_retries = 3
# loop if response is 429 or 500-599 3 times
    while (
        response.status_code == 429
        or 500 <= response.status_code <= 599
    ) and retry_count < max_retries:

        wait_seconds = 2 ** retry_count 

        print(
            f"{location}: HTTP {response.status_code}. "
            f"Retrying in {wait_seconds} seconds..."
        )

        time.sleep(wait_seconds)

        response = requests.get(url, params=params, timeout=30)
        retry_count += 1

    city.append(location)
    request_url.append(safe_url)
    http_status.append(response.status_code)

    try:
        raw_payload.append(json.dumps(response.json()))
    except ValueError:
        raw_payload.append(response.text)

    ingestion_timestamp.append(datetime.now(timezone.utc))

bronze_record = [
    {
        "city": city_value,
        "request_url": url_value,
        "http_status": status_value,
        "raw_payload": payload_value,
        "ingestion_timestamp": timestamp_value
    }
    for city_value, url_value, status_value, payload_value, timestamp_value
    in zip(
        city,
        request_url,
        http_status,
        raw_payload,
        ingestion_timestamp
    )
]

bronze_df = spark.createDataFrame(bronze_record)
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog}.{schema}.weather_bronze")

## 3. Create the Silver Table

Create `weather_silver` as a Delta table to store parsed, cleaned, and deduplicated weather forecasts.

`IF NOT EXISTS` creates the table only when it does not already exist, preserving existing data.

| Column | Description |
|---|---|
| `city` | City associated with the forecast. |
| `forecast_unix_timestamp` | Forecast time in seconds since the Unix epoch. |
| `forecast_timestamp` | Forecast time converted to a timestamp. |
| `temperature` | Temperature in Celsius. |
| `feels_like` | Perceived temperature in Celsius. |
| `humidity` | Relative humidity (%). |
| `pressure` | Atmospheric pressure (hPa). |
| `weather_main` | General weather condition, such as Clear, Clouds, or Rain. |
| `weather_description` | Detailed weather description. |
| `wind_speed` | Wind speed in meters per second. |
| `wind_direction` | Wind direction in degrees. |
| `ingestion_timestamp` | UTC timestamp when the API response was collected. |
| `weather_hash` | Hash of the weather values, used to detect changes during MERGE. |

The intended record key is `(city, forecast_unix_timestamp)`. Deduplication and MERGE logic maintain one row per key; this table definition does not enforce uniqueness.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_demo_catalog.weather_analytics.weather_silver(
city string, 
forecast_unix_timestamp bigint,
forecast_timestamp timestamp,
 temperature double,
 feels_like double,
 humidity int,
 pressure int,
 weather_main string,
 weather_description string,
 wind_speed double, 
 wind_direction int, 
 ingestion_timestamp timestamp,
 weather_hash BIGINT)
USING DELTA;

## 4. Transform and Merge Forecasts into Silver

Parse the Bronze JSON payloads, standardize weather values, and merge the latest forecasts into `weather_silver`.

### Parse and Transform

- Process successful API responses (`http_status = 200`).
- Use `from_json()` to parse the forecast array and `explode()` to create one row per forecast.
- Convert epoch seconds into timestamps using UTC.
- Convert Kelvin temperatures to Celsius and round to two decimal places.
- Exclude records with a missing city or forecast epoch timestamp.

### Keep the Latest Forecast

Use `ROW_NUMBER()` partitioned by `(city, forecast_unix_timestamp)`, ordered by `ingestion_timestamp DESC`.

Keeping row number `1` selects the most recently collected forecast for each city and forecast time.

### Detect Changes Using a Hash

A later API call can return the same city and forecast time with revised weather values, such as a different temperature, humidity, or wind speed.

Calculate `weather_hash` using `xxhash64()` over the changeable weather columns. Comparing the stored and incoming hashes provides a compact change check instead of writing a separate comparison for every weather column.

Exclude `ingestion_timestamp` from the hash because a new collection time alone does not mean the forecast changed.

Hash comparison simplifies the update condition, but it does not guarantee faster execution, and hash collisions are possible.

### Apply the MERGE

| Situation | Action |
|---|---|
| City and forecast time do not exist | Insert the forecast. |
| Key exists, hash differs, and incoming ingestion is at least as recent | Update the forecast. |
| Key exists and hash is unchanged | Keep the existing row. |
| Incoming ingestion is older | Keep the existing row. |

The null-safe comparison `<=>` also allows existing rows with a null hash to be updated.

`ingestion_timestamp` in Silver changes only when an insert or weather-value update occurs.

In [0]:
%sql
-- Make timestamp conversions display in UTC.
SET TIME ZONE 'UTC';

MERGE INTO dev_demo_catalog.weather_analytics.weather_silver AS target
USING (
    WITH forecasts AS (
        SELECT
            city,
            ingestion_timestamp,
            explode(
                from_json(
                    raw_payload:list,
                    'ARRAY<STRUCT<
                        dt: BIGINT,
                        dt_txt: STRING,
                        main: STRUCT<
                            temp: DOUBLE,
                            feels_like: DOUBLE,
                            humidity: INT,
                            pressure: INT
                        >,
                        weather: ARRAY<STRUCT<
                            main: STRING,
                            description: STRING
                        >>,
                        wind: STRUCT<
                            speed: DOUBLE,
                            deg: INT
                        >
                    >>'
                )
            ) AS forecast
        FROM dev_demo_catalog.weather_analytics.weather_bronze
        WHERE http_status = 200
    ),

    parsed_weather AS (
        SELECT
            city,
            forecast.dt AS forecast_unix_timestamp,
            timestamp_seconds(forecast.dt) AS forecast_timestamp,

            -- Standard API units are Kelvin.
            round(forecast.main.temp - 273.15,2) AS temperature,
            round(forecast.main.feels_like - 273.15,2) AS feels_like,

            forecast.main.humidity AS humidity,
            forecast.main.pressure AS pressure,
            forecast.weather[0].main AS weather_main,
            forecast.weather[0].description AS weather_description,
            forecast.wind.speed AS wind_speed,
            forecast.wind.deg AS wind_direction,
            ingestion_timestamp
        FROM forecasts
        WHERE city IS NOT NULL
          AND forecast.dt IS NOT NULL
    ),

    forecasts_window AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY city, forecast_unix_timestamp
                ORDER BY ingestion_timestamp DESC
            ) AS ingestion_number
        FROM parsed_weather
    )

    SELECT
        *,
        xxhash64(
            temperature,
            feels_like,
            humidity,
            pressure,
            weather_main,
            weather_description,
            wind_speed,
            wind_direction
        ) AS weather_hash
    FROM forecasts_window
    WHERE ingestion_number = 1
) AS source

ON  target.city = source.city
AND target.forecast_unix_timestamp = source.forecast_unix_timestamp

WHEN MATCHED
    AND NOT (target.weather_hash <=> source.weather_hash)
    AND source.ingestion_timestamp >= target.ingestion_timestamp
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *;

## 5. Create the Gold Daily Weather Summary

Create or replace `weather_gold_summary` as a view containing **one row per city per forecast date**, ready for the AI/BI dashboard.

### Aggregate Daily Metrics

Convert `forecast_timestamp` to a date and group the Silver records by city and date.

| Column | Description |
|---|---|
| `city` | Forecast city. |
| `forecast_date` | Forecast date, based on the session timezone. |
| `avg_temperature_celsius` | Average temperature (°C). |
| `min_temperature_celsius` | Minimum temperature (°C). |
| `max_temperature_celsius` | Maximum temperature (°C). |
| `avg_humidity` | Average relative humidity (%). |
| `max_wind_speed_mps` | Maximum wind speed (m/s). |
| `data_point_count` | Number of forecast records in the group. |
| `most_frequent_weather_main` | Most common non-null weather condition. |

### Identify the Dominant Condition

Use `MODE(weather_main, TRUE)` to return the most frequent weather condition for each city and day.

`TRUE` makes tie selection deterministic. If all conditions in a group are null, the result is null.

### Keep Results Connected to Silver

The view stores the query rather than a separate copy of the results. Querying it calculates the summary from the current Silver data.

In [0]:
%sql
CREATE OR REPLACE VIEW
    dev_demo_catalog.weather_analytics.weather_gold_summary AS

SELECT
    city,
    CAST(forecast_timestamp AS DATE) AS forecast_date,

    -- Daily temperature metrics in Celsius.
    AVG(temperature) AS avg_temperature_celsius,
    MIN(temperature) AS min_temperature_celsius,
    MAX(temperature) AS max_temperature_celsius,

    -- Daily humidity and wind metrics.
    AVG(humidity) AS avg_humidity,
    MAX(wind_speed) AS max_wind_speed_mps,

    -- Count the forecast records for each city and day.
    COUNT(*) AS data_point_count,

    -- Find the most frequent non-null weather condition.
    -- TRUE makes the selection deterministic when frequencies tie.
    MODE(weather_main, TRUE) AS most_frequent_weather_main

FROM dev_demo_catalog.weather_analytics.weather_silver
GROUP BY
    city,
    CAST(forecast_timestamp AS DATE);